# Allergen Detection from Ingredient Lists
## Multi-label Classification with MobileBERT

This notebook trains a multi-label classifier to detect 8 allergens from product ingredient texts.

## Overview

Trains MobileBERT model for multi-label allergen classification.

## Workflow

1. Load and prepare data using utility modules
2. Split into train/val/test with stratified splitting
3. Train model with weighted loss
4. Evaluate and compute optimal thresholds
5. Save final model

## 1. Setup & Imports

Using modular utility functions to reduce code duplication

In [1]:
import os
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score
from scipy.special import expit

# Import project utility modules
from utils.data_utils import (
    get_data_directories,
    load_labeled_data,
    create_stratified_splits,
    augment_dataframe
)
from utils.text_processing import (
    clean_html,
    get_allergen_list,
    allergens_to_binary
)
from utils.model_utils import (
    compute_class_weights,
    find_best_thresholds,
    apply_thresholds
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis
)

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("All imports completed using utility modules.")

All imports completed using utility modules.


## 2. Load and Prepare Data

In [2]:
# Get project directories for consistent path handling
dirs = get_data_directories()
print(f"Project base directory: {dirs['base']}")

# Load labeled data using utility function
# Use the enhanced dataset which contains detected allergens
df = load_labeled_data(filepath=os.path.join(dirs['final'], 'labeled_dataset_enhanced.csv'))
print(f"Dataset shape: {df.shape}")
df.head()

Project base directory: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION
Dataset shape: (1057, 11)


,code,brands,product_name_en,ingredients_text_en,detected_allergens,official_allergens_mapped,traces_allergens,consensus_allergens,combined_allergens,may_contain,coconut
0,4806502720615,Gardenia,Gardenia White Bread Classic 600G,"wheat flour (vitamin b2, vitamin b3, vitamin a...","['milk', 'wheat']","['milk', 'wheat']",['soy'],"['milk', 'wheat']","{'detected_only': [], 'detected_or_official': ...",[],['coconut']
1,2116136,Signature Select,Slightly Dipped Almonds: French Vanilla Dark C...,"almonds, semi-sweet chocolate (unsweetened cho...",['milk'],"['soy', 'tree_nuts']",[],[],"{'detected_only': ['milk'], 'detected_or_offic...",[],[]
2,4801981120147,Minute Maid,Orange Juice Drink,"water, sugar, acidulant (citric acid), orange ...",[],[],[],[],"{'detected_only': [], 'detected_or_official': ...",[],[]
3,4800194178167,Oishi,Oishi Spicy Seafood Curls Shrimp Flavored Snack,"tapioca starch, vegetable oil, coconut oil, pa...","['shellfish', 'wheat']","['shellfish', 'wheat']",[],"['shellfish', 'wheat']","{'detected_only': [], 'detected_or_official': ...",[],['coconut']
4,5000159558914,Snickers,Snickers Miniatures,"sugar, glucose syrup, peanuts, skimmed milk po...","['milk', 'peanuts', 'eggs', 'soy']","['milk', 'peanuts', 'eggs', 'soy']",['tree_nuts'],"['eggs', 'milk', 'peanuts', 'soy']","{'detected_only': [], 'detected_or_official': ...",[],[]


In [3]:
# Clean HTML tags from ingredients using utility function
df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)

# Convert detected allergens list to binary vector
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

print(f"Sample cleaned text: {df['ingredients_cleaned'].iloc[0][:200]}")
print(f"Label distribution (per class): {np.array(df['labels'].tolist()).sum(axis=0)}")

Sample cleaned text: wheat flour (vitamin b2, vitamin b3, vitamin a), purified water, refined sugar, baker's yeast, whey powder, iodized salt, skimmed milk powder, dextrose, pure vegetable shortening ( palm oil, coconut o
Label distribution (per class): [495 102  89  15 277 358  77  33]


## 3. Train/Validation/Test Split (Stratified)

Using utility function for multilabel stratified split

In [4]:
# Prepare data for splitting
X = df["ingredients_cleaned"].values
y = np.array(df["labels"].tolist())

# Create stratified splits using utility function
# Using 70% train, 15% validation, 15% test
train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=SEED
)

print(f"Train size: {len(train_texts)} (positive samples: {np.array(train_labels).sum(axis=0).sum()})")
print(f"Val size:   {len(val_texts)}   (positive samples: {np.array(val_labels).sum(axis=0).sum()})")
print(f"Test size:  {len(test_texts)}  (positive samples: {np.array(test_labels).sum(axis=0).sum()})")

Train size: 739 (positive samples: 1010)
Val size:   159   (positive samples: 215)
Test size:  159  (positive samples: 221)


## 4. Data Augmentation (Training Set Only)

Using utility function for data augmentation

In [5]:
# Create temporary dataframes for augmentation
train_df = pd.DataFrame({
    'text': train_texts,
    'labels': train_labels.tolist(),
    'ingredients_cleaned': train_texts  # Simplified for augmentation
})

# Apply data augmentation using utility function
train_aug_df = augment_dataframe(train_df, num_augmented=2)
print(f"Original training size: {len(train_df)}")
print(f"Augmented training size: {len(train_aug_df)}")

# Combine original training + augmented training
combined_train_df = pd.concat([train_df, train_aug_df], ignore_index=True)
print(f"Combined training size: {len(combined_train_df)}")

# Extract texts and labels for training
train_texts_final = combined_train_df["text"].tolist()
train_labels_final = np.array(combined_train_df["labels"].tolist())

# Validation and test sets remain unchanged
val_texts_final = val_texts
val_labels_final = np.array(val_labels)
test_texts_final = test_texts
test_labels_final = np.array(test_labels)

Original training size: 739
Augmented training size: 1242
Combined training size: 1981


## 5. Compute Class Weights for Imbalance

Using utility function to compute class weights

In [6]:
# Compute class weights using utility function
class_weights_tensor = compute_class_weights(train_labels_final)

# Validate training data: check for NaN/Inf in labels
assert not np.any(np.isnan(train_labels_final)), "NaN found in training labels!"
assert not np.any(np.isinf(train_labels_final)), "Inf found in training labels!"
assert not np.any(np.isnan(val_labels_final)), "NaN found in validation labels!"
assert not np.any(np.isinf(val_labels_final)), "Inf found in validation labels!"

print("Training data validation passed: no NaN/Inf values.")
print("Class weights:")
allergen_list = get_allergen_list()
for i, allergen in enumerate(allergen_list):
    print(f"  {allergen:10s}: {class_weights_tensor[i]:.3f}")

Training data validation passed: no NaN/Inf values.
Class weights:
  milk      : 0.367
  eggs      : 0.995
  peanuts   : 1.071
  tree_nuts : 1.892
  soy       : 0.584
  wheat     : 0.492
  fish      : 1.113
  shellfish : 1.485


## 6. Tokenization

Determine optimal max_length based on 95th percentile of token lengths

In [7]:
# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "google/mobilebert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Compute max length from training texts
sample_lengths = [len(tokenizer.encode(t, truncation=False)) for t in train_texts_final[:500]]
max_len = int(np.percentile(sample_lengths, 95))
MAX_LEN = min(max_len, 256)
print(f"Using max_length = {MAX_LEN}")

def tokenize_data(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_encodings = tokenize_data(train_texts_final)
val_encodings = tokenize_data(val_texts_final)
test_encodings = tokenize_data(test_texts_final)

class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = FoodDataset(train_encodings, train_labels_final)
val_dataset = FoodDataset(val_encodings, val_labels_final)
test_dataset = FoodDataset(test_encodings, test_labels_final)

# Ensure training output directory exists using dirs for consistency
output_dir = os.path.join(dirs['base'], "models", "mobilebert_allergen")
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready: {output_dir}")

Using device: cuda
Using max_length = 219
Output directory ready: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen


## 7. Model Definition & Training Utilities

Using manual PyTorch training loop (HuggingFace Trainer causes segfault
in this environment's PyTorch 2.12.0+cu130 build).

In [8]:
# Load model
n_classes = len(get_allergen_list())
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=n_classes,
    problem_type="multi_label_classification"
)
model = model.to(device)

# NOTE: MobileBERT's pooler output has extreme magnitudes (~±54M) because the
# encoder hidden states are unconstrained. This causes epoch 1 BCE loss >1M
# with the default nn.Linear classifier. Our experiments show that normalizing
# the pooler output with LayerNorm prevents learning entirely on this small
# dataset (1962 augmented samples). The high epoch 1 loss is cosmetic — the
# model still converges to Macro F1 ~0.86 (see NB08 cell-17 for the baseline).
#
# We therefore keep the original nn.Linear classifier and accept the high
# initial loss. Gradient clipping (max_norm=1.0) handles backward stability.

print(f"Model loaded on {device}")
print(f"  Classifier: nn.Linear (original, no LayerNorm)")
print(f"  Note: epoch 1 loss >1M is expected and harmless")

class_weights_tensor = class_weights_tensor.to(device)

def run_train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device, epoch_num=None):
    """Run one training epoch, returns average loss.
    
    Logs gradient norms on epoch 1, first batch to catch loss explosion early.
    """
    model.train()
    total_loss = 0.0
    for batch_idx, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = loss_fn(outputs.logits, batch["labels"])
        loss.backward()
        
        # Gradient monitoring: log max gradient norm on epoch 1, batch 0
        if epoch_num == 1 and batch_idx == 0:
            total_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2).item()
                    total_norm += param_norm ** 2
            total_norm = total_norm ** 0.5
            print(f"  [Monitor] Epoch 1, Batch 0 — gradient norm: {total_norm:.2f}")
            if loss.item() > 100:
                print(f"  [Monitor] Epoch 1, Batch 0 — loss: {loss.item():.2f} (expected with MobileBERT pooler)")
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def evaluate(model, dataloader, loss_fn, device):
    """Evaluate model, returns (avg_loss, macro_f1, logits_array, labels_array)."""
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = loss_fn(outputs.logits, batch["labels"])
            total_loss += loss.item()
            all_logits.append(outputs.logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = expit(logits)
    preds = (probs > 0.5).astype(int)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return total_loss / len(dataloader), f1, logits, labels


def predict(model, dataset, device, batch_size=8):
    """Run inference and return (logits, labels) arrays."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)


print("Training utilities ready.")

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on cuda
  Classifier: nn.Linear (original, no LayerNorm)
  Note: epoch 1 loss >1M is expected and harmless
Training utilities ready.


## 8. Training Configuration

Hyperparameters for manual training loop.

In [9]:
# Training hyperparameters
batch_size = 8
num_epochs = 20
learning_rate = 2e-5
weight_decay = 0.01
max_grad_norm = 1.0
early_stopping_patience = 3

train_steps_per_epoch = max(1, len(train_texts_final) // batch_size)
total_training_steps = train_steps_per_epoch * num_epochs

# === FIX: Ensure at least 200 warmup steps ===
# Linear warmup from lr=0 for the first N steps prevents the optimizer from
# amplifying the initial extreme logits from the randomly-initialized head.
warmup_steps = max(200, int(0.1 * total_training_steps))

print(f"Samples: {len(train_texts_final)} train, {len(val_texts_final)} val")
print(f"Steps per epoch: {train_steps_per_epoch}")
print(f"Total training steps: {total_training_steps}, warmup: {warmup_steps}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Loss function with class weights
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor, reduction='mean')

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# LR scheduler with linear warmup + linear decay
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Training configuration ready.")

Samples: 1981 train, 159 val
Steps per epoch: 247
Total training steps: 4940, warmup: 494
Training configuration ready.


In [10]:
%%time
print("=" * 60)
print("Starting training")
print("=" * 60)

best_val_f1 = 0.0
best_model_state = None
no_improve_epochs = 0
train_losses, val_losses, val_f1s = [], [], []

output_dir = os.path.join(dirs['base'], "models", "mobilebert_allergen")
os.makedirs(output_dir, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    # Train for one epoch (pass epoch_num for gradient monitoring)
    train_loss = run_train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device, epoch_num=epoch)
    
    # Evaluate on validation set
    val_loss, val_f1, val_logits, val_labels = evaluate(model, val_loader, loss_fn, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)
    
    # Early stopping check
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve_epochs = 0
        # Save checkpoint for best model
        torch.save(model.state_dict(), os.path.join(output_dir, "best_model.pt"))
        tokenizer.save_pretrained(output_dir)
    else:
        no_improve_epochs += 1
    
    print(f"Epoch {epoch:2d}/{num_epochs}  "
          f"Train Loss: {train_loss:.4f}  "
          f"Val Loss: {val_loss:.4f}  "
          f"Val Macro F1: {val_f1:.4f}  "
          f"{'*' if val_f1 == best_val_f1 and epoch > 1 else ''}"
          f"{' (no improv. ' + str(no_improve_epochs) + ')' if no_improve_epochs > 0 else ''}")
    
    if no_improve_epochs >= early_stopping_patience:
        print(f"\nEarly stopping triggered after {epoch} epochs.")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    model = model.to(device)

print(f"\nTraining complete. Best Val Macro F1: {best_val_f1:.4f}")
print(f"Training history saved in train_losses, val_losses, val_f1s arrays")

Starting training
  [Monitor] Epoch 1, Batch 0 — gradient norm: 112716729.58
  [Monitor] Epoch 1, Batch 0 — loss: 4319003.50 (expected with MobileBERT pooler)
Epoch  1/20  Train Loss: 1332176.2340  Val Loss: 1216.4910  Val Macro F1: 0.0158  
Epoch  2/20  Train Loss: 4.7600  Val Loss: 5.5765  Val Macro F1: 0.6139  *
Epoch  3/20  Train Loss: 0.0806  Val Loss: 0.8192  Val Macro F1: 0.7673  *
Epoch  4/20  Train Loss: 0.2242  Val Loss: 0.8756  Val Macro F1: 0.7808  *
Epoch  5/20  Train Loss: 0.0757  Val Loss: 0.0756  Val Macro F1: 0.7915  *
Epoch  6/20  Train Loss: 0.5634  Val Loss: 1.5810  Val Macro F1: 0.7968  *
Epoch  7/20  Train Loss: 0.0165  Val Loss: 2.3357  Val Macro F1: 0.7962   (no improv. 1)
Epoch  8/20  Train Loss: 0.0111  Val Loss: 0.1126  Val Macro F1: 0.7972  *
Epoch  9/20  Train Loss: 0.0084  Val Loss: 0.0730  Val Macro F1: 0.8215  *
Epoch 10/20  Train Loss: 0.0085  Val Loss: 0.1097  Val Macro F1: 0.8009   (no improv. 1)
Epoch 11/20  Train Loss: 0.0070  Val Loss: 0.1069  Val 

## 9. Evaluate on Validation Set (Baseline with 0.5 threshold)

In [11]:
# Use manual predict instead of trainer.predict
val_logits, val_output_labels = predict(model, val_dataset, device, batch_size=batch_size)
val_probs = expit(val_logits)
val_preds_05 = (val_probs >= 0.5).astype(int)

print("Baseline (threshold=0.5) Macro F1:",
      f1_score(val_output_labels, val_preds_05, average="macro", zero_division=0))

Baseline (threshold=0.5) Macro F1: 0.8264187240658867


## 10. Per-Class Threshold Optimization

We find optimal thresholds on validation set to maximize macro F1 per class
Using utility function for threshold optimization

In [12]:
# Find optimal thresholds using utility function
best_thresholds = find_best_thresholds(val_probs, val_labels_final)

# Save thresholds to both .npy and hybrid_config.json for consistency
np.save(os.path.join(dirs['models'], "best_thresholds.npy"), best_thresholds)

# Also save to hybrid_config.json so it becomes the single source of truth
import json
hybrid_config = {
    "ml_thresholds": best_thresholds.tolist(),
    "rule_conf_threshold": 0.5,
    "mode": "hard_override"
}
config_path = os.path.join(dirs["models"], "hybrid_config.json")
with open(config_path, "w") as f:
    json.dump(hybrid_config, f, indent=2)
print(f"Hybrid config saved to {config_path}")

print("Optimal thresholds:")
allergen_list = get_allergen_list()
for i, allergen in enumerate(allergen_list):
    print(f"{allergen:12s} → threshold={best_thresholds[i]:.2f}")

Hybrid config saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/hybrid_config.json
Optimal thresholds:
milk         → threshold=0.07
eggs         → threshold=0.04
peanuts      → threshold=0.96
tree_nuts    → threshold=0.50
soy          → threshold=0.33
wheat        → threshold=0.05
fish         → threshold=0.30
shellfish    → threshold=0.08


## 11. Apply Optimized Thresholds

Using utility function to apply thresholds

In [13]:
# Apply optimized thresholds using utility function
val_preds_opt = apply_thresholds(val_probs, best_thresholds)

print("\nValidation set after threshold tuning:")
print(f"Macro F1: {f1_score(val_labels_final, val_preds_opt, average='macro', zero_division=0):.4f}")
print(f"Micro F1: {f1_score(val_labels_final, val_preds_opt, average='micro', zero_division=0):.4f}")


Validation set after threshold tuning:
Macro F1: 0.8332
Micro F1: 0.9650


## 12. Final Evaluation on Test Set

Using utility functions for evaluation and error analysis

In [14]:
# Use manual predict instead of trainer.predict
test_logits, test_labels_true = predict(model, test_dataset, device, batch_size=batch_size)
test_probs = expit(test_logits)
test_preds = apply_thresholds(test_probs, best_thresholds)

print("\n=== Test Set Performance ===")
print_classification_report(test_labels_true, test_preds, get_allergen_list(), prefix="Test Set")

# Optional: Per-class metrics
print_per_class_metrics(test_labels_true, test_preds, get_allergen_list(), prefix="Per-class metrics on test set:")

# Optional: Error analysis
error_indices = error_analysis(test_texts_final, test_labels_true, test_preds, get_allergen_list(), max_errors=3)


=== Test Set Performance ===
=== Test Set ===
              precision    recall  f1-score   support

        milk     0.9600    0.9600    0.9600        75
        eggs     0.9375    0.9375    0.9375        16
     peanuts     1.0000    0.9286    0.9630        14
   tree_nuts     0.0000    0.0000    0.0000         3
         soy     0.9333    1.0000    0.9655        42
       wheat     0.9608    0.9074    0.9333        54
        fish     0.9167    0.9167    0.9167        12
   shellfish     1.0000    1.0000    1.0000         5

   micro avg     0.9539    0.9367    0.9452       221
   macro avg     0.8385    0.8313    0.8345       221
weighted avg     0.9415    0.9367    0.9386       221
 samples avg     0.5920    0.5956    0.5890       221

Macro F1: 0.8345
Micro F1: 0.9452


Per-class metrics on test set:
milk          Prec: 96.00%  Rec: 96.00%  F1: 96.00%
eggs          Prec: 93.75%  Rec: 93.75%  F1: 93.75%
peanuts       Prec: 100.00%  Rec: 92.86%  F1: 96.30%
tree_nuts     Prec: 0.00

## 13. Save Final Model

Saving model and tokenizer

In [15]:
final_dir = os.path.join(dirs['models'], "mobilebert_allergen_final")
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)

# The classifier is a standard nn.Linear (no LayerNorm wrapper), so
# AutoModelForSequenceClassification.from_pretrained() loads correctly.
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Model saved to {final_dir}")

Model saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_final
